# A3 delta-stepping testbench

1. **Runtime → Change runtime type → T4 GPU**.
2. Run the cells top to bottom. The upload cell asks for `CS23B078.cu`.
3. Each test runs your binary several times and compares it with a CPU reference (Dijkstra + the spec's adaptive delta log), so it checks correctness **and** determinism.

Timings include reading the input file (the boilerplate `infile >>` parsing), so big tests look slower than the GPU work alone.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!free -g | head -2
!df -h /content | tail -1

In [ ]:
import os
from google.colab import files
if not os.path.exists('/content/CS23B078.cu'):
    files.upload()  # pick CS23B078.cu
!ls -la /content/CS23B078.cu
!mkdir -p testbench

In [ ]:
%%writefile testbench/gen.cpp
// Graph generator for the A3 delta-stepping testbench.
// Usage: gen <out> <N> <E> <S> <delta> <K> <type> <wmin> <wmax> <seed>
//   type: rand  - uniform random edges
//         hub   - ~30% of edges leave 8 hub vertices, ~10% of targets are hubs (skewed degrees)
//         chain - path 0->1->...->N-1 plus short forward edges (long diameter, many buckets)
//         sparse- many vertices with no out-edges, many unreachable ones
// Edges are produced directly in CSR order, so memory is O(N + E) with small constants.
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cstdint>
#include <vector>
#include <random>
#include <string>

static FILE* fout;
static char buf[1 << 22];
static size_t bpos = 0;

static inline void flush_buf() { fwrite(buf, 1, bpos, fout); bpos = 0; }
static inline void put_int(long long x, char sep) {
    if (bpos + 24 > sizeof(buf)) flush_buf();
    if (x < 0) { buf[bpos++] = '-'; x = -x; }
    char tmp[24]; int n = 0;
    do { tmp[n++] = '0' + x % 10; x /= 10; } while (x);
    while (n) buf[bpos++] = tmp[--n];
    buf[bpos++] = sep;
}

int main(int argc, char** argv) {
    if (argc < 11) {
        fprintf(stderr, "usage: gen out N E S delta K type wmin wmax seed\n");
        return 1;
    }
    const char* out = argv[1];
    int N = atoi(argv[2]);
    long long E = atoll(argv[3]);
    int S = atoi(argv[4]);
    int delta = atoi(argv[5]);
    int K = atoi(argv[6]);
    std::string type = argv[7];
    int wmin = atoi(argv[8]), wmax = atoi(argv[9]);
    uint64_t seed = strtoull(argv[10], nullptr, 10);

    std::mt19937_64 rng(seed);
    auto rnd = [&](uint64_t n) { return (uint64_t)(rng() % n); };

    // ---- degrees ----
    std::vector<long long> deg(N, 0);
    const int HUBS = std::min(8, N);
    if (type == "rand") {
        for (long long i = 0; i < E; i++) deg[rnd(N)]++;
    } else if (type == "hub") {
        long long hubE = E * 3 / 10;
        for (long long i = 0; i < hubE; i++) deg[rnd(HUBS)]++;
        for (long long i = hubE; i < E; i++) deg[rnd(N)]++;
    } else if (type == "chain") {
        long long chainE = std::min<long long>(E, N - 1);
        for (long long u = 0; u < chainE; u++) deg[u]++;
        for (long long i = chainE; i < E; i++) deg[rnd(N)]++;
    } else if (type == "sparse") {
        // only ~1/4 of the vertices have out-edges
        int active = std::max(1, N / 4);
        for (long long i = 0; i < E; i++) deg[rnd(active) * 4 % N]++;
    } else {
        fprintf(stderr, "unknown type %s\n", type.c_str());
        return 1;
    }

    // ---- edges in CSR order ----
    std::vector<int> neighs(E);
    std::vector<uint16_t> weights(E);
    long long idx = 0;
    for (int u = 0; u < N; u++) {
        for (long long j = 0; j < deg[u]; j++) {
            int v;
            if (type == "hub") {
                v = (rnd(10) == 0) ? (int)rnd(HUBS) : (int)rnd(N);
            } else if (type == "chain") {
                if (j == 0 && u < N - 1) v = u + 1;
                else v = (int)((u + 1 + rnd(64)) % N);
            } else if (type == "sparse") {
                v = (int)rnd(N);
                if (v % 2 == 1) v = (v + 1) % N;   // odd vertices mostly unreachable
            } else {
                v = (int)rnd(N);
            }
            neighs[idx] = v;
            weights[idx] = (uint16_t)(wmin + rnd(wmax - wmin + 1));
            idx++;
        }
    }

    // ---- sources: vertex 0 first (interesting for chain), the rest random ----
    std::vector<int> src(S);
    for (int i = 0; i < S; i++) src[i] = (i == 0) ? 0 : (int)rnd(N);
    if (type == "hub" && S > 1) src[1] = 1 % N;  // a hub as source
    if (type == "hub" && S > 2) src[2] = N - 1;  // a likely ordinary vertex

    fout = fopen(out, "wb");
    if (!fout) { perror(out); return 1; }
    put_int(delta, ' '); put_int(K, '\n');
    put_int(N, ' '); put_int(E, ' '); put_int(S, '\n');
    for (int i = 0; i < S; i++) put_int(src[i], i + 1 == S ? '\n' : ' ');
    long long off = 0;
    for (int u = 0; u <= N; u++) {
        put_int(off, u == N ? '\n' : ' ');
        if (u < N) off += deg[u];
    }
    for (long long i = 0; i < E; i++) put_int(neighs[i], i + 1 == E ? '\n' : ' ');
    if (E == 0) buf[bpos++] = '\n';
    for (long long i = 0; i < E; i++) put_int(weights[i], i + 1 == E ? '\n' : ' ');
    flush_buf();
    fclose(fout);
    return 0;
}


In [ ]:
%%writefile testbench/ref.cpp
// CPU reference for the A3 testbench.
// Usage: ref <input> <output>
// Distances come from Dijkstra. In adaptive mode (delta == -1) the delta log is
// reproduced from the spec (section 11): after each bucket, every vertex with
// dist <= cutoff is settled, the Far keys are the best tentative distances through
// settled vertices, and
//   Dmin = min key, cutoff = key[min(K-1, |far|-1)], log cutoff - Dmin.
// The first batch (cutoff = 0) is not logged.
#include <cstdio>
#include <cstdlib>
#include <cstdint>
#include <vector>
#include <queue>
#include <set>
#include <algorithm>
#include <functional>

static const int INF = 2147483647;

static std::vector<char> data;
static size_t pos = 0;
static long long read_ll() {
    while (pos < data.size() && (data[pos] == ' ' || data[pos] == '\n' || data[pos] == '\r' || data[pos] == '\t')) pos++;
    bool neg = false;
    if (data[pos] == '-') { neg = true; pos++; }
    long long x = 0;
    while (pos < data.size() && data[pos] >= '0' && data[pos] <= '9') x = x * 10 + (data[pos++] - '0');
    return neg ? -x : x;
}

static char obuf[1 << 22];
static size_t opos = 0;
static FILE* fout;
static void put_int(long long x) {
    if (opos + 24 > sizeof(obuf)) { fwrite(obuf, 1, opos, fout); opos = 0; }
    if (x < 0) { obuf[opos++] = '-'; x = -x; }
    char t[24]; int n = 0;
    do { t[n++] = '0' + x % 10; x /= 10; } while (x);
    while (n) obuf[opos++] = t[--n];
    obuf[opos++] = '\n';
}

int main(int argc, char** argv) {
    if (argc < 3) { fprintf(stderr, "usage: ref in out\n"); return 1; }
    FILE* fin = fopen(argv[1], "rb");
    if (!fin) { perror(argv[1]); return 1; }
    fseek(fin, 0, SEEK_END); long long sz = ftell(fin); fseek(fin, 0, SEEK_SET);
    data.resize(sz + 1);
    if (fread(data.data(), 1, sz, fin) != (size_t)sz) { fprintf(stderr, "read error\n"); return 1; }
    data[sz] = '\n';
    fclose(fin);

    int delta = (int)read_ll(), K = (int)read_ll();
    int N = (int)read_ll(); long long E = read_ll(); int S = (int)read_ll();
    std::vector<int> src(S);
    for (auto& s : src) s = (int)read_ll();
    std::vector<long long> off(N + 1);
    for (auto& o : off) o = read_ll();
    std::vector<int> nb(E), w(E);
    for (auto& x : nb) x = (int)read_ll();
    for (auto& x : w) x = (int)read_ll();
    std::vector<char>().swap(data);

    fout = fopen(argv[2], "wb");
    if (!fout) { perror(argv[2]); return 1; }

    std::vector<int> d(N), tent(N), order;
    std::vector<char> settled(N);
    for (int s : src) {
        // ---- Dijkstra ----
        std::fill(d.begin(), d.end(), INF);
        d[s] = 0;
        using P = std::pair<int, int>;
        std::priority_queue<P, std::vector<P>, std::greater<P>> pq;
        pq.push({0, s});
        while (!pq.empty()) {
            auto [du, u] = pq.top(); pq.pop();
            if (du != d[u]) continue;
            for (long long i = off[u]; i < off[u + 1]; i++) {
                int nd = du + w[i];
                if (nd < d[nb[i]]) { d[nb[i]] = nd; pq.push({nd, nb[i]}); }
            }
        }

        put_int(s);

        // ---- adaptive delta log ----
        if (delta == -1) {
            order.clear();
            for (int v = 0; v < N; v++) if (d[v] != INF) order.push_back(v);
            std::sort(order.begin(), order.end(), [&](int a, int b) { return d[a] < d[b]; });
            std::fill(tent.begin(), tent.end(), INF);
            std::fill(settled.begin(), settled.end(), 0);
            std::set<P> far;  // (tent, vertex) of reached, unsettled vertices
            size_t p = 0;
            long long cutoff = 0;
            std::vector<int> batch;
            while (true) {
                batch.clear();
                while (p < order.size() && d[order[p]] <= cutoff) {
                    int v = order[p++];
                    settled[v] = 1;
                    if (tent[v] != INF) far.erase({tent[v], v});
                    batch.push_back(v);
                }
                for (int u : batch) {
                    for (long long i = off[u]; i < off[u + 1]; i++) {
                        int v = nb[i];
                        if (settled[v]) continue;
                        int nd = d[u] + w[i];
                        if (nd < tent[v]) {
                            if (tent[v] != INF) far.erase({tent[v], v});
                            tent[v] = nd;
                            far.insert({nd, v});
                        }
                    }
                }
                if (far.empty()) break;
                int Dmin = far.begin()->first;
                auto it = far.begin();
                for (int k = 0; k < K - 1 && std::next(it) != far.end(); k++) ++it;
                cutoff = it->first;
                put_int(cutoff - Dmin);
            }
        }

        for (int v = 0; v < N; v++) put_int(d[v]);
    }
    fwrite(obuf, 1, opos, fout);
    fclose(fout);
    return 0;
}


In [ ]:
%%writefile testbench/run_bench.py
#!/usr/bin/env python3
"""A3 delta-stepping testbench.

Builds the generator, the CPU reference and the solution, generates heavy and
edge-case graphs, and runs the solution several times per test to check
correctness (vs. the reference) and determinism (all runs identical).

  python3 run_bench.py --sol CS23B078.cu                 # default suite
  python3 run_bench.py --sol CS23B078.cu --only a_k1 s_delta0
  python3 run_bench.py --sol CS23B078.cu --huge          # adds E = 300M tests
  python3 run_bench.py --sol CS23B078.cu --runs 5 --timeout 900
"""
import argparse
import filecmp
import os
import subprocess
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))

# name: (N, E, S, delta, K, type, wmin, wmax)
TESTS = {
    # ---- static ----
    "s_rand_big":    (2_000_000, 40_000_000, 3,   50,     -1, "rand",   0, 1000),
    "s_hub":         (1_000_000, 20_000_000, 3,  100,     -1, "hub",    1, 1000),
    "s_delta0":      (  200_000,  2_000_000, 2,    0,     -1, "rand",   0,   20),
    "s_delta1_chain":(  100_000,    400_000, 2,    1,     -1, "chain",  1,   10),
    "s_bigdelta":    (2_000_000, 30_000_000, 2, 1000,     -1, "rand",   0, 1000),
    "s_sparse":      (1_000_000,  5_000_000, 3,   10,     -1, "sparse", 0,  100),
    "s_allzero":     (  500_000,  5_000_000, 2,    5,     -1, "rand",   0,    0),
    "s_20src":       (  100_000,  1_000_000, 20,   7,     -1, "rand",   0, 1000),
    # ---- adaptive ----
    "a_rand_big":    (2_000_000, 40_000_000, 3,   -1,  65536, "rand",   0, 1000),
    "a_hub_plateau": (1_000_000, 20_000_000, 3,   -1,   1024, "hub",    0,    5),
    "a_kmax":        (2_000_000, 30_000_000, 2,   -1, 524288, "rand",   1, 1000),
    "a_k1":          (   20_000,    200_000, 2,   -1,      1, "rand",   0,    3),
    "a_chain":       (  200_000,  1_000_000, 2,   -1,     16, "chain",  1,  100),
    "a_sparse":      (1_000_000,  5_000_000, 3,   -1,   4096, "sparse", 0,   10),
    "a_zero_one":    (  500_000,  5_000_000, 2,   -1,     64, "rand",   0,    1),
    "a_20src":       (  100_000,  1_000_000, 20,  -1,    256, "hub",    0, 1000),
    # ---- edge cases ----
    "e_single_s":    (        1,          0, 2,    3,     -1, "rand",   0,    5),
    "e_single_a":    (        1,          0, 2,   -1,      5, "rand",   0,    5),
    "e_selfloops_a": (    1_000,     20_000, 20,  -1,      7, "hub",    0,   10),
    "e_k_gt_n":      (    1_000,      5_000, 3,   -1, 524288, "rand",   0, 1000),
}

HUGE = {
    "h_static":      (2_000_000, 300_000_000, 1,  100,    -1, "rand",   0, 1000),
    "h_adaptive":    (2_000_000, 300_000_000, 1,   -1, 65536, "rand",   0, 1000),
}

GREEN, RED, YELLOW, NC = "\033[0;32m", "\033[0;31m", "\033[1;33m", "\033[0m"


def sh(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, **kw)


def first_diff(a, b, limit=5):
    """Return (count of differing lines, list of first few diffs), ignoring blank lines."""
    def lines(path):
        with open(path) as f:
            for i, line in enumerate(f, 1):
                s = line.strip()
                if s:
                    yield i, s
    diffs, count = [], 0
    ia, ib = lines(a), lines(b)
    while True:
        x, y = next(ia, None), next(ib, None)
        if x is None and y is None:
            break
        xs = x[1] if x else "<EOF>"
        ys = y[1] if y else "<EOF>"
        if xs != ys:
            count += 1
            if len(diffs) < limit:
                diffs.append((x[0] if x else y[0], xs, ys))
    return count, diffs


def same(a, b):
    if filecmp.cmp(a, b, shallow=False):
        return True
    return first_diff(a, b, limit=0)[0] == 0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sol", required=True, help="path to the .cu solution")
    ap.add_argument("--runs", type=int, default=3)
    ap.add_argument("--timeout", type=int, default=600, help="seconds per run")
    ap.add_argument("--only", nargs="*", help="run only these tests")
    ap.add_argument("--huge", action="store_true", help="also run E = 300M tests")
    ap.add_argument("--workdir", default=os.path.join(HERE, "work"))
    args = ap.parse_args()

    os.makedirs(args.workdir, exist_ok=True)
    gen = os.path.join(args.workdir, "gen")
    ref = os.path.join(args.workdir, "ref")
    sol = os.path.join(args.workdir, "main")
    sh(["g++", "-O2", "-std=c++17", "-o", gen, os.path.join(HERE, "gen.cpp")])
    sh(["g++", "-O2", "-std=c++17", "-o", ref, os.path.join(HERE, "ref.cpp")])
    sh(["nvcc", args.sol, "-o", sol])  # same command the graders use

    tests = dict(TESTS)
    if args.huge:
        tests.update(HUGE)
    if args.only:
        missing = [t for t in args.only if t not in tests]
        if missing:
            sys.exit(f"unknown tests: {missing} (huge tests need --huge)")
        tests = {t: tests[t] for t in args.only}

    results = []
    for seed, (name, (N, E, S, delta, K, typ, wmin, wmax)) in enumerate(tests.items(), 1):
        print(f"\n=== {name}: N={N:,} E={E:,} S={S} delta={delta} K={K} type={typ} w=[{wmin},{wmax}]", flush=True)
        inp = os.path.join(args.workdir, name + ".in")
        exp = os.path.join(args.workdir, name + ".exp")
        if not os.path.exists(inp):
            t0 = time.time()
            sh([gen, inp, str(N), str(E), str(S), str(delta), str(K), typ, str(wmin), str(wmax), str(seed)])
            print(f"  generated in {time.time() - t0:.1f}s", flush=True)
        if not os.path.exists(exp):
            t0 = time.time()
            sh([ref, inp, exp])
            print(f"  reference in {time.time() - t0:.1f}s", flush=True)

        outs, times, status = [], [], "PASS"
        for r in range(args.runs):
            out = os.path.join(args.workdir, f"{name}.run{r}.out")
            t0 = time.time()
            try:
                p = subprocess.run([sol, inp, out], capture_output=True, text=True, timeout=args.timeout)
            except subprocess.TimeoutExpired:
                print(f"  run {r}: {RED}TIMEOUT{NC} (> {args.timeout}s)")
                status = "TIMEOUT"
                break
            dt = time.time() - t0
            times.append(dt)
            if p.returncode != 0:
                print(f"  run {r}: {RED}CRASH{NC} exit={p.returncode}\n  stderr: {p.stderr.strip()[:500]}")
                status = "CRASH"
                break
            if p.stderr.strip():
                print(f"  run {r} stderr: {p.stderr.strip()[:500]}")
            outs.append(out)
            if same(out, exp):
                print(f"  run {r}: {GREEN}OK{NC} {dt:.2f}s", flush=True)
            else:
                cnt, diffs = first_diff(out, exp)
                print(f"  run {r}: {RED}WRONG{NC} {dt:.2f}s, {cnt} differing lines; first ones (line: got | expected):")
                for ln, got, want in diffs:
                    print(f"    {ln}: {got} | {want}")
                status = "WRONG"

        # determinism across runs (matters even when all are wrong)
        nondet = len(outs) > 1 and any(not same(outs[0], o) for o in outs[1:])
        if nondet:
            print(f"  {YELLOW}NON-DETERMINISTIC{NC}: runs produced different outputs")
            if status == "PASS":
                status = "WRONG"
        results.append((name, status, times, nondet))

        # keep disk usage down on Colab: drop outputs of passing runs
        if status == "PASS":
            for o in outs:
                os.remove(o)

    print("\n" + "=" * 64)
    print(f"{'test':<18}{'result':<10}{'min s':>8}{'max s':>8}  notes")
    for name, status, times, nondet in results:
        color = GREEN if status == "PASS" else RED
        tmin = f"{min(times):.2f}" if times else "-"
        tmax = f"{max(times):.2f}" if times else "-"
        note = "non-deterministic" if nondet else ""
        print(f"{name:<18}{color}{status:<10}{NC}{tmin:>8}{tmax:>8}  {note}")
    passed = sum(s == "PASS" for _, s, _, _ in results)
    print(f"\n{passed}/{len(results)} passed")


if __name__ == "__main__":
    main()


## 1. Quick tests (small / edge cases, ~1 min)
Run these first: if they fail, the big ones will too.

In [ ]:
!python3 -u testbench/run_bench.py --sol CS23B078.cu --runs 3 --only e_single_s e_single_a e_selfloops_a e_k_gt_n a_k1 s_delta0 s_delta1_chain a_chain s_20src a_20src

## 2. Memory checker on small inputs
`compute-sanitizer` reports out-of-bounds accesses and failed CUDA calls. It's slow, so only small inputs. A clean run ends with `ERROR SUMMARY: 0 errors`.

In [ ]:
for t in ['e_single_a', 'e_selfloops_a', 'a_k1', 's_delta0']:
    print('==', t)
    !compute-sanitizer --tool memcheck testbench/work/main testbench/work/{t}.in /tmp/san.out 2>&1 | tail -3

## 3. Full heavy suite (N up to 2M, E up to 40M)
Generating inputs takes a few minutes the first time; they're reused on later runs.

In [ ]:
!python3 -u testbench/run_bench.py --sol CS23B078.cu --runs 3

## 4. Optional: full-scale E = 300M
~3.5 GB input files and several minutes per step. Close to Colab's RAM limit; skip it if the session restarts.

In [ ]:
!python3 -u testbench/run_bench.py --sol CS23B078.cu --runs 1 --timeout 1800 --huge --only h_static h_adaptive

## Re-running after editing the .cu
Upload the new file over `/content/CS23B078.cu` (the file pane works), then re-run a test cell: the solution is recompiled each time; generated inputs and references are reused.

Free disk space: `!rm -rf testbench/work/*.in testbench/work/*.exp`